# 008 - Brute Force QUBO portfolio Notebook 

## Admin 

In [1]:
# Load Libraties 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qubo_functions import (
    compute_expected_return_vector,
    compute_covariance_matrix,
    build_qubo_matrix_mean_variance,
    generate_all_binary_portfolios,
    evaluate_all_binary_portfolios,
    find_optimal_portfolio_by_energy,
    export_qubo_terms_for_qaoa,
)

import environment as envir 
from tabulate import tabulate 

In [2]:
data, headers = envir.environment_state()
print(tabulate(data, headers=headers, tablefmt="grid"))

+-------------------------+-----------+
| System/Library          | Version   |
+=========================+===========+
| Python                  | 3.10.9    |
+-------------------------+-----------+
| numpy                   | 2.2.6     |
+-------------------------+-----------+
| torch                   | 2.9.1     |
+-------------------------+-----------+
| matplotlib              | 3.10.8    |
+-------------------------+-----------+
| seaborn                 | 0.13.2    |
+-------------------------+-----------+
| qiskit                  | 1.4.4     |
+-------------------------+-----------+
| qiskit_machine_learning | 0.8.4     |
+-------------------------+-----------+
| yfinance                | 1.2.0     |
+-------------------------+-----------+


### Step 1 - Create Assets with Synthetic Returns Data 

In [3]:
# Replace this synthetic block with your real data load
Number_of_Assets = 5
Number_of_Time_Periods = 252

np.random.seed(42)
Synthetic_Returns_Array = np.random.normal(
    loc=0.0005,
    scale=0.01,
    size=(Number_of_Time_Periods, Number_of_Assets),
)

Asset_Names = [f"Asset_{i}" for i in range(Number_of_Assets)]

Asset_Returns_DataFrame = pd.DataFrame(
    Synthetic_Returns_Array,
    columns=Asset_Names,
)

display(Asset_Returns_DataFrame.head())

,Asset_0,Asset_1,Asset_2,Asset_3,Asset_4
0,0.005467,-0.000883,0.006977,0.015730,-0.001842
1,-0.001841,0.016292,0.008174,-0.004195,0.005926
2,-0.004134,-0.004157,0.002920,-0.018633,-0.016749
3,-0.005123,-0.009628,0.003642,-0.008580,-0.013623
4,0.015156,-0.001758,0.001175,-0.013747,-0.004944


### Step 2 - Compute the Expected Returns and Covariance Matrix 

In [4]:
Expected_Return_Vector = compute_expected_return_vector(
    Asset_Returns_DataFrame=Asset_Returns_DataFrame
)

Sigma_Covariance_Matrix = compute_covariance_matrix(
    Asset_Returns_DataFrame=Asset_Returns_DataFrame
)

print("Expected_Return_Vector:")
print(Expected_Return_Vector)
print("\nSigma_Covariance_Matrix:")
print(Sigma_Covariance_Matrix)

Expected_Return_Vector:
[0.00064493 0.00018952 0.00049786 0.00134388 0.00172974]

Sigma_Covariance_Matrix:
[[ 8.76922006e-05 -5.60715876e-06  8.16079504e-06  1.07627800e-07
   2.37331562e-06]
 [-5.60715876e-06  1.07802937e-04  7.14434533e-07 -2.70197492e-06
   7.57409034e-06]
 [ 8.16079504e-06  7.14434533e-07  9.37394707e-05 -9.46021126e-07
   8.36954925e-06]
 [ 1.07627800e-07 -2.70197492e-06 -9.46021126e-07  9.43151018e-05
  -5.48318475e-06]
 [ 2.37331562e-06  7.57409034e-06  8.36954925e-06 -5.48318475e-06
   1.05951268e-04]]


### Step 3 - Build the QUBO Matrix 

In [5]:
Risk_Aversion_Parameter = 10.0  # adjust as desired

Q_Matrix = build_qubo_matrix_mean_variance(
    Sigma_Covariance_Matrix=Sigma_Covariance_Matrix,
    Expected_Return_Vector=Expected_Return_Vector,
    Risk_Aversion_Parameter=Risk_Aversion_Parameter,
)

print("\nQ_Matrix:")
print(Q_Matrix)


Q_Matrix:
[[ 2.31991606e-04 -5.60715876e-05  8.16079504e-05  1.07627800e-06
   2.37331562e-05]
 [-5.60715876e-05  8.88511028e-04  7.14434533e-06 -2.70197492e-05
   7.57409034e-05]
 [ 8.16079504e-05  7.14434533e-06  4.39530020e-04 -9.46021126e-06
   8.36954925e-05]
 [ 1.07627800e-06 -2.70197492e-05 -9.46021126e-06 -4.00732046e-04
  -5.48318475e-05]
 [ 2.37331562e-05  7.57409034e-05  8.36954925e-05 -5.48318475e-05
  -6.70228326e-04]]


### Step 4 - Generate all binary portfolios 

In [6]:
Binary_Portfolios = generate_all_binary_portfolios(
    Number_of_Assets=Number_of_Assets
)

print(f"\nNumber of binary portfolios: {len(Binary_Portfolios)}")


Number of binary portfolios: 32


### Step 5 - Evaluate all portfolios, risk, and returns 

In [7]:
Evaluation_DataFrame = evaluate_all_binary_portfolios(
    Binary_Portfolios=Binary_Portfolios,
    Q_Matrix=Q_Matrix,
    Expected_Return_Vector=Expected_Return_Vector,
    Sigma_Covariance_Matrix=Sigma_Covariance_Matrix,
    Asset_Names=Asset_Names,
)

display(Evaluation_DataFrame.head())

,Binary_Portfolio,Energy,Portfolio_Return,Portfolio_Risk,Asset_0_Included,Asset_1_Included,Asset_2_Included,Asset_3_Included,Asset_4_Included
0,"(0, 0, 0, 0, 0)",0.000000,0.000000,0.000000,0,0,0,0,0
1,"(0, 0, 0, 0, 1)",-0.000670,0.001730,0.000106,0,0,0,0,1
2,"(0, 0, 0, 1, 0)",-0.000401,0.001344,0.000094,0,0,0,1,0
3,"(0, 0, 0, 1, 1)",-0.001181,0.003074,0.000189,0,0,0,1,1
4,"(0, 0, 1, 0, 0)",0.000440,0.000498,0.000094,0,0,1,0,0


### Step 6 - Identified optimized portfolio by minumum energy 

In [8]:
Optimal_Portfolio_Row = find_optimal_portfolio_by_energy(
    Evaluation_DataFrame=Evaluation_DataFrame
)

print("\nOptimal portfolio by energy:")
display(Optimal_Portfolio_Row)

print("\nOptimal binary portfolio:", Optimal_Portfolio_Row["Binary_Portfolio"])
print("Optimal energy:", Optimal_Portfolio_Row["Energy"])
print("Optimal portfolio return:", Optimal_Portfolio_Row["Portfolio_Return"])
print("Optimal portfolio risk:", Optimal_Portfolio_Row["Portfolio_Risk"]) 


Optimal portfolio by energy:


Binary_Portfolio    (0, 0, 0, 1, 1)
Energy                    -0.001181
Portfolio_Return           0.003074
Portfolio_Risk             0.000189
Asset_0_Included                  0
Asset_1_Included                  0
Asset_2_Included                  0
Asset_3_Included                  1
Asset_4_Included                  1
Name: 3, dtype: object


Optimal binary portfolio: (0, 0, 0, 1, 1)
Optimal energy: -0.0011806240671568807
Optimal portfolio return: 0.003073624069195235
Optimal portfolio risk: 0.00018930000020383548


### Step 7 - Visualization of responses

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(Evaluation_DataFrame["Energy"], bins=20, edgecolor="black")
plt.xlabel("Energy")
plt.ylabel("Count")
plt.title("Distribution of QUBO Energy Values")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(
    Evaluation_DataFrame["Portfolio_Risk"],
    Evaluation_DataFrame["Portfolio_Return"],
    c=Evaluation_DataFrame["Energy"],
    cmap="viridis",
    alpha=0.7,
)
plt.colorbar(label="Energy")
plt.xlabel("Portfolio Risk (Variance)")
plt.ylabel("Portfolio Expected Return")
plt.title("Risk–Return–Energy Landscape")
plt.tight_layout()
plt.show() 

### Step 8 = Export the QUBO terms for a QAOA analysis 

In [ ]:
QUBO_Terms_Dictionary = export_qubo_terms_for_qaoa(
    Q_Matrix=Q_Matrix
)

print("\nLinear terms (first few):")
for k in list(QUBO_Terms_Dictionary["linear_terms"].keys())[:5]:
    print(k, ":", QUBO_Terms_Dictionary["linear_terms"][k])

print("\nQuadratic terms (first few):")
for k in list(QUBO_Terms_Dictionary["quadratic_terms"].keys())[:5]:
    print(k, ":", QUBO_Terms_Dictionary["quadratic_terms"][k])

Linear_Terms_DataFrame = pd.DataFrame(
    list(QUBO_Terms_Dictionary["linear_terms"].items()),
    columns=["Asset_Index", "Linear_Coefficient"]
)

Quadratic_Terms_DataFrame = pd.DataFrame(
    [(i, j, value) for (i, j), value in QUBO_Terms_Dictionary["quadratic_terms"].items()],
    columns=["Asset_i", "Asset_j", "Quadratic_Coefficient"]
)

QUBO_Linear_Terms_Out = "../data/QUBO_linear_terms.csv"
QUBO_Quadratic_Terms_Out = "../data/QUBO_quadratic_terms.csv"


Linear_Terms_DataFrame.to_csv(QUBO_Linear_Terms_Out, index=False)
Quadratic_Terms_DataFrame.to_csv(QUBO_Quadratic_Terms_Out, index=False)
